# 11 - Hive

Paste each Bash block into an Ubuntu terminal in order. Keep one terminal for this notebook. Keep HDFS and MySQL running. The small table example loads a file and uses a fetch query; it does not require a running YARN job.

## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Services: hive-metastore:9083 and hiveserver2:10000. JVM metrics: port 9404 on both Services. MySQL hive_metastore stores metadata on mysql-data. Table data lives in HDFS on the HDFS PVCs. data-hive-metastore-0 and data-hiveserver2-0 are 5 GiB each at /data; Hive log files are forwarded from /tmp/team by file-log sidecars. MySQL, HDFS and the metrics ConfigMap are prerequisites.

## Check stores

```bash
kubectl rollout status statefulset/mysql --timeout=600s
kubectl rollout status statefulset/namenode --timeout=600s
kubectl exec namenode-0 -- hdfs dfsadmin -safemode wait
```



## Prepare the metastore database

```bash
kubectl exec mysql-0 -- sh -ec 'MYSQL_PWD="$MYSQL_ROOT_PASSWORD" mysql -h127.0.0.1 -uroot -e "CREATE DATABASE IF NOT EXISTS hive_metastore; GRANT ALL ON hive_metastore.* TO '\''team'\''@'\''%'\'';"'
```



## Prepare the schema job

```bash
kubectl apply -f monitoring/telemetry-config.yaml
if kubectl get job hive-schema >/dev/null 2>&1; then
  kubectl wait --for=condition=complete job/hive-schema --timeout=900s
  kubectl delete job hive-schema
fi
kubectl apply -f hive/schema.yaml
```

Only the completed schema validation Job is replaced. Its code initializes an empty schema and verifies an existing schema.

## Wait for the schema

```bash
kubectl wait --for=condition=complete job/hive-schema --timeout=900s
kubectl logs job/hive-schema
```



## Apply Hive servers

```bash
kubectl apply -f hive/hive.yaml
```



## Wait for hive-metastore

```bash
kubectl rollout status statefulset/hive-metastore --timeout=900s
```



## Wait for hiveserver2

```bash
kubectl rollout status statefulset/hiveserver2 --timeout=900s
```



## Prepare an HDFS input file

```bash
printf '1,workspace hive example\n' > .state/test-hive.csv
kubectl exec namenode-0 -- hdfs dfs -mkdir -p /user/team/test-hive
kubectl exec -i namenode-0 -- hdfs dfs -put -f - /user/team/test-hive/input.csv < .state/test-hive.csv
```



## Create and read the Hive table

```bash
kubectl exec hiveserver2-0 -c hiveserver2 -- beeline -u jdbc:hive2://hiveserver2:10000 -n team --silent=true --outputformat=tsv2 -e "SET hive.fetch.task.conversion=more; CREATE DATABASE IF NOT EXISTS test_hive; CREATE TABLE IF NOT EXISTS test_hive.test_table (id INT, message STRING) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' STORED AS TEXTFILE; LOAD DATA INPATH '/user/team/test-hive/input.csv' OVERWRITE INTO TABLE test_hive.test_table; SELECT * FROM test_hive.test_table;" > .state/test-hive-query.txt
cat .state/test-hive-query.txt
grep 'workspace hive example' .state/test-hive-query.txt
```



## Inspect Hive storage

```bash
kubectl get pvc mysql-data data-hive-metastore-0 data-hiveserver2-0
kubectl exec namenode-0 -- hdfs dfs -ls /user/hive/warehouse/test_hive.db/test_table
```



## Check collected logs and metrics

```bash
python3 validations/check-observability.py --components hive-metastore,hiveserver2 --jobs hive-metastore-jmx,hiveserver2-jmx
```

The helper makes temporary loopback API forwards, waits for actual log/metric samples and closes its forwards. It fails if collection is missing. Allow several scrape intervals after startup.

## Check Hive file logs

```bash
python3 validations/check-observability.py --components hiveserver2 --file-logs
```



## Stop Hive

```bash
bash stop-components.sh hive
```

Retains MySQL metadata, HDFS table files and PVCs.

## Resume Hive

```bash
bash start-components.sh hive
```



## Read the retained table

```bash
kubectl exec hiveserver2-0 -c hiveserver2 -- beeline -u jdbc:hive2://hiveserver2:10000 -n team --silent=true --outputformat=tsv2 -e 'SET hive.fetch.task.conversion=more; SELECT * FROM test_hive.test_table;' > .state/test-hive-persisted.txt
grep 'workspace hive example' .state/test-hive-persisted.txt
```

